# Deep Research Agent with Oracle AI Agent Memory

[![Documentation](https://img.shields.io/badge/Documentation-Oracle%20AI%20Agent%20Memory-red?style=flat-square)](https://www.oracle.com/database/ai-agent-memory/)


**Framework:** OpenAI Agents SDK (driven by Claude via LiteLLM) · **Web search:** Tavily · **Memory:** Oracle AI Agent Memory on Oracle AI Database

This notebook walks through building a **deep research agent** for human genome exploration. The agent uses Tavily for live web research, and stores its running conversation and durable findings in Oracle AI Database via the `oracleagentmemory` package.

---

## Application Modes in Agent Applications

AI agent applications generally fall into three operational modes:

| Mode | Shape | Typical use |
|---|---|---|
| **Assistant** | Turn-by-turn conversational | Customer support, coding copilot, chat UIs |
| **Deep Research** | Multi-step autonomous investigation | Literature review, market research, technical scoping |
| **Workflow** | Predetermined sequence of steps, often with conditional branches | Approval pipelines, compliance checks, structured triage |

> **📌 This notebook focuses on the Deep Research mode.**
>
> A deep-research agent plans, retrieves, synthesises, and accumulates findings over long-horizon investigations. Memory is central — without durable memory, every session restarts from zero and the agent cannot build on prior research.

![Application modes](images/application_mode.png)

![The application-mode spectrum](images/application_mode_spectrum.png)


## What You'll Learn

- How to implement the **OpenAI Agents SDK `Session` protocol** against a custom backend (Oracle AI Agent Memory)
- How to wrap **Tavily** as a `function_tool` the agent can call
- How to store long-lived research findings as durable **memories** (separate from short-term conversation history)
- How to verify memory continuity across sessions — the agent can pick up where it left off

> **💡 Key insight:** Short-term memory (conversation history for the current run) and long-term memory (durable findings across runs) are different access patterns over the same store. We use `Session` for the former and `add_memory()` for the latter.

## Prerequisites

- **Oracle AI Database** running locally (Docker container) or reachable over the network
- **`ANTHROPIC_API_KEY`** for the agent (Claude, via LiteLLM) and OAMP's helper LLM. Embeddings run locally via `nomic-embed-text-v1.5` (fastembed) — no OpenAI key needed
- **`TAVILY_API_KEY`** for web search (free tier available at [tavily.com](https://tavily.com))
- The `oracleagentmemory` wheel installed in the active kernel's environment

## 1. Install dependencies

We need four packages: `oracleagentmemory` + `fastembed` (memory + local nomic embeddings), `openai-agents` (agent framework), `tavily-python` (web search), and `nest_asyncio` (so `Runner.run` works cleanly inside a Jupyter event loop).

In [1]:
# Core memory library + OpenAI Agents SDK (driven by Claude via LiteLLM) + Tavily + local embeddings.
# No OpenAI key is needed; openai-agents just needs the `openai` package present.
%pip install -q oracleagentmemory openai-agents tavily-python nest_asyncio fastembed
%pip install -q --upgrade "openai>=2.26,<3"

Note: you may need to restart the kernel to use updated packages.


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
litellm 1.83.14 requires openai==2.24.0, but you have openai 2.41.1 which is incompatible.


Note: you may need to restart the kernel to use updated packages.


## 2. Configure API keys and Oracle connection

Set environment variables for Anthropic, Tavily, and the Oracle database. We use `os.environ.setdefault` so shell or `.env` values still win if you've set them elsewhere.

In [ ]:
import os
import getpass

# LLM + embedding provider (used by both the agent and OAMP)

# Tavily for web search
if not os.environ.get("TAVILY_API_KEY"):
    os.environ["TAVILY_API_KEY"] = getpass.getpass("Enter your Tavily API Key: ")

if not os.environ.get("ANTHROPIC_API_KEY"):
    os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("Enter your ANTHROPIC_API_KEY: ")
# Oracle AI Database connection
os.environ.setdefault("DB_USER", "VECTOR")
os.environ.setdefault("DB_PASSWORD", "VectorPwd_2025")
os.environ.setdefault("DB_CONNECT_STRING", "localhost:1521/FREEPDB1")

# Jupyter runs an async event loop already — this lets us call async SDK methods cleanly
import nest_asyncio
nest_asyncio.apply()

print("Environment configured.")

Environment configured.


## 3. Connect to Oracle AI Database and initialise the memory client

`OracleAgentMemory` is the governed memory client. With `extract_memories=True` and an `llm` attached, the client will automatically extract durable memories from messages added to a thread. We use `text-embedding-3-small` for semantic search over the memory store.

> **💡 Key insight:** `schema_policy` controls how the library interacts with the DB on startup. `REQUIRE_EXISTING` (default) never issues DDL — safe for production. `CREATE_IF_NECESSARY` fills in missing objects — safe for dev. `RECREATE` drops and rebuilds — useful when you've changed embedding dimensions.

In [3]:
import oracledb
import numpy as np
from fastembed import TextEmbedding
from oracleagentmemory.core import OracleAgentMemory
from oracleagentmemory.core.llms import Llm
from oracleagentmemory.apis.embedders.embedder import IEmbedder


def _unit(v):
    v = np.asarray(v, dtype=np.float32)
    n = np.linalg.norm(v)
    return v / n if n else v


class NomicEmbedder(IEmbedder):
    """Open-source, local embeddings: nomic-embed-text-v1.5 (768-dim) via fastembed ONNX.

    No API key and no network call — this is what lets the notebook run without OpenAI.
    fastembed applies nomic's `search_document:` / `search_query:` prefixes internally
    (passages via `embed`, queries via `query_embed`).
    """

    def __init__(self, model_name="nomic-ai/nomic-embed-text-v1.5"):
        self._model = TextEmbedding(model_name=model_name)

    def embed(self, texts, *, is_query=False):
        gen = self._model.query_embed(texts) if is_query else self._model.embed(texts)
        return np.asarray([_unit(v) for v in gen], dtype=np.float32)

    async def embed_async(self, texts, *, is_query=False):
        return self.embed(texts, is_query=is_query)


connection = oracledb.connect(
    user=os.environ["DB_USER"],
    password=os.environ["DB_PASSWORD"],
    dsn=os.environ["DB_CONNECT_STRING"],
)
print("Connected to Oracle AI Database:", connection.version)

# Local open-source embedder (no OpenAI) + Claude Haiku as the lightweight
# extraction / summary LLM (routed through litellm's `anthropic/` provider).
embedder = NomicEmbedder()
extraction_llm = Llm("anthropic/claude-sonnet-4-6", temperature=0.2)

memory_client = OracleAgentMemory(
    connection=connection,
    embedder=embedder,
    llm=extraction_llm,
    extract_memories=True,
    schema_policy="recreate",
    table_name_prefix="genome_",
)
print("Memory client ready (nomic embeddings + Claude helper — no OpenAI).")

/Users/richmondalake/opt/anaconda3/envs/oracle_demos/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Connected to Oracle AI Database: 23.26.0.0.0


Memory client ready (nomic embeddings + Claude helper — no OpenAI).


## 4. Register the research user and agent

Oracle AI Agent Memory scopes every record by `user_id` and `agent_id`. Registering them up-front gives the store a stable identity to hang memories on and lets us enforce tenant-style isolation across multiple users.

> **📌 Scoping matters for production.** In a multi-user deployment, `user_id` should match your application's authenticated user. The memory client's `search()` method requires a concrete `user_id` on every call — an intentional guardrail against cross-tenant leaks.

In [4]:
USER_ID = "genome-researcher-1"
AGENT_ID = "deep-research-agent"

for create_fn, eid, info in [
    (memory_client.add_user, USER_ID, "Genomics researcher exploring human disease associations."),
    (memory_client.add_agent, AGENT_ID, "Deep-research agent with web search and long-term memory."),
]:
    try:
        create_fn(eid, info)
        print(f"Registered: {eid}")
    except ValueError:
        print(f"Already exists: {eid}")

Registered: genome-researcher-1
Registered: deep-research-agent


## 5. Implement the `Session` protocol on top of Oracle AI Agent Memory

The OpenAI Agents SDK defines a `Session` protocol with four async methods:

| Method | Purpose |
|---|---|
| `get_items(limit)` | Return the conversation history the runner should inject |
| `add_items(items)` | Persist new items the runner produced during a run |
| `pop_item()` | Remove and return the most recent item (for corrections) |
| `clear_session()` | Drop everything for this session |

Implementing this protocol against Oracle AI Agent Memory gives us **exact-resumption** short-term memory — the next time this session runs, the agent receives the full prior conversation without any manual wiring.

> **💡 Key insight:** The `Session` protocol handles *short-term* memory (the items the runner replays each turn). Long-term memory (durable facts the agent should remember across sessions) is a separate concern that we handle with `add_memory()` below.

In [5]:
import json
from typing import Optional
from agents.memory.session import SessionABC


class OracleAgentMemorySession(SessionABC):
    """Session backend that persists OpenAI Agents SDK items in Oracle AI Agent Memory.

    Each item is serialised to JSON and stored as a memory record tagged with the
    session's thread_id. Ordering is preserved by the store's insertion timestamp.
    """

    def __init__(self, session_id: str, client: OracleAgentMemory, user_id: str, agent_id: str):
        self.session_id = session_id
        self._client = client
        self._store = client._store
        self._user_id = user_id
        self._agent_id = agent_id
        # Ensure the underlying thread exists so memories can attach to it
        try:
            self._client.create_thread(
                thread_id=session_id, user_id=user_id, agent_id=agent_id,
            )
        except ValueError:
            pass  # thread already exists

    async def get_items(self, limit: Optional[int] = None) -> list:
        records = self._store.list(
            "memory",
            thread_id=self.session_id,
            user_id=self._user_id,
            agent_id=self._agent_id,
            limit=limit if limit else 10_000,
            metadata_filter={"session_item": True},
        )
        items = [json.loads(r.content) for r in records]
        return items[-limit:] if limit else items

    async def add_items(self, items: list) -> None:
        if not items:
            return
        for item in items:
            self._client.add_memory(
                json.dumps(item),
                user_id=self._user_id,
                agent_id=self._agent_id,
                thread_id=self.session_id,
                metadata={"session_item": True},
            )

    async def pop_item(self) -> Optional[dict]:
        records = self._store.list(
            "memory",
            thread_id=self.session_id,
            user_id=self._user_id,
            agent_id=self._agent_id,
            limit=1,
            metadata_filter={"session_item": True},
        )
        if not records:
            return None
        # Records come back newest-first when limit=1 is applied; delete and return
        last = records[-1]
        self._store.delete("memory", last.id)
        return json.loads(last.content)

    async def clear_session(self) -> None:
        records = self._store.list(
            "memory",
            thread_id=self.session_id,
            user_id=self._user_id,
            agent_id=self._agent_id,
            limit=10_000,
            metadata_filter={"session_item": True},
        )
        for r in records:
            self._store.delete("memory", r.id)


print("OracleAgentMemorySession implemented.")

OracleAgentMemorySession implemented.


## 6. Define the agent's tools

The agent needs three tools:

1. **`tavily_search`** — live web search for up-to-date genomics sources
2. **`save_research_finding`** — persist a durable finding as a **`fact` record** (semantic memory) the agent can recall in future sessions
3. **`recall_research_findings`** — search that semantic memory for prior findings on a topic

Tools 2 and 3 are how the agent manages its **semantic** memory — consolidated knowledge that outlives any single conversation. They're distinct from the `Session` machinery (next section), which handles the **episodic** conversation history automatically.

> **📌 Two record types, two memory tiers.** Storing findings as `fact` (via the store's typed `add`) keeps durable knowledge separate from the raw episodic trail. `recall_research_findings` filters `record_types=["fact"]` and returns each hit's `formatted_content` — the prompt-ready rendering the agent reads back.

In [6]:
from agents import function_tool
from tavily import TavilyClient

_tavily = TavilyClient(api_key=os.environ["TAVILY_API_KEY"])

@function_tool
def tavily_search(query: str, max_results: int = 5) -> str:
    """Search the live web for recent, authoritative genomics sources.

    Use this to pull current information from NCBI, OMIM, peer-reviewed journals,
    and other scientific sources. Prefer this over relying on model-internal knowledge
    for facts about specific genes, mutations, or recent studies.

    Args:
        query: Natural-language search query.
        max_results: How many results to return (1-10).
    """
    resp = _tavily.search(
        query=query, max_results=max_results,
        search_depth="advanced", include_answer=True,
    )
    lines = [f"Answer: {resp.get('answer', '')}"]
    for r in resp.get("results", []):
        lines.append(f"- {r['title']} ({r['url']})\n  {r['content'][:300]}")
    return "\n".join(lines)


@function_tool
def save_research_finding(topic: str, finding: str) -> str:
    """Persist a durable research finding the agent should recall in future sessions.

    Findings are stored as `fact` records — the agent's semantic memory (consolidated
    knowledge), kept distinct from the episodic conversation history. Use this for claims
    worth remembering across sessions — e.g. "BRCA1 mutations are associated with elevated
    lifetime breast cancer risk." Do not use this for conversational acknowledgements.

    Args:
        topic: Short topic tag (e.g. 'BRCA1', 'TP53 mutations').
        finding: The finding to remember, ideally one sentence.
    """
    ids = memory_client._store.add(
        contents=[f"[{topic}] {finding}"],
        record_type="fact",                      # semantic memory
        user_ids=USER_ID,
        agent_ids=AGENT_ID,
        metadata={"topic": topic, "kind": "research_finding"},
    )
    return f"Saved finding (id={ids[0]})."


@function_tool
def recall_research_findings(query: str, max_results: int = 5) -> str:
    """Search prior research findings (semantic memory) for information relevant to a query.

    Use this at the start of a research task to check what is already known.
    Results are ranked by semantic similarity and returned in prompt-ready form.

    Args:
        query: Natural-language query describing what you want to recall.
        max_results: How many findings to return.
    """
    results = memory_client.search(
        query, user_id=USER_ID, agent_id=AGENT_ID,
        record_types=["fact"], max_results=max_results,
    )
    if not results:
        return "(no prior findings)"
    # formatted_content is the prompt-ready, type-tagged rendering of each hit.
    return "\n".join(r.formatted_content for r in results)


print("Tools defined: tavily_search, save_research_finding, recall_research_findings")

Tools defined: tavily_search, save_research_finding, recall_research_findings


## 7. Construct the research agent

The agent's `instructions` are its system prompt — the place to encode the behaviour you want. For a deep-research agent, the instructions should emphasise:

1. **Recall before research** — check long-term memory before hitting the web
2. **Cite sources** — so findings are traceable
3. **Save what's worth remembering** — commit durable conclusions explicitly

> **💡 Key insight:** A deep-research agent's output quality is largely determined by how well its instructions distinguish *research* (retrieve + synthesise) from *conversation* (respond + acknowledge). Encode that distinction in the system prompt.

In [7]:
from agents import Agent, Runner
from agents.extensions.models.litellm_model import LitellmModel

INSTRUCTIONS = """You are a deep-research agent specialising in human genome exploration.

For every research question:
1. FIRST call `recall_research_findings` to check what is already known from prior sessions.
2. If the prior findings are insufficient or outdated, call `tavily_search` for current sources.
3. Synthesise a clear, cited answer. Prefer authoritative sources (NCBI, OMIM, PubMed).
4. Call `save_research_finding` for each durable conclusion worth remembering across sessions.
   Save one finding per call; keep findings atomic and one sentence long.
5. Present the final answer to the user with inline citations (URLs).

Do not save conversational acknowledgements as findings. Only save factual conclusions.
"""

research_agent = Agent(
    name="GenomeDeepResearcher",
    instructions=INSTRUCTIONS,
    tools=[tavily_search, save_research_finding, recall_research_findings],
    model=LitellmModel(model="anthropic/claude-opus-4-8", api_key=os.environ["ANTHROPIC_API_KEY"]),
)
print(f"Agent created: {research_agent.name}")

Agent created: GenomeDeepResearcher


## 8. Run a research session

We create a session (tied to a `thread_id` in the memory store) and run the agent over a sequence of research questions. Because we pass `session=session`, the SDK will automatically inject prior conversation items at the start of each turn and persist new items at the end.

> **📌 Separation of concerns.**
> - The `Session` persists the **raw conversation** (working memory).
> - The agent's `save_research_finding` tool persists **durable findings** (long-term memory).
> Both live in the same Oracle database but are accessed through different interfaces.

In [8]:
SESSION_ID = "genome-session-001"
session = OracleAgentMemorySession(
    session_id=SESSION_ID,
    client=memory_client,
    user_id=USER_ID,
    agent_id=AGENT_ID,
)

research_questions = [
    "What is BRCA1 and what is its primary function in DNA repair?",
    "What is the typical lifetime breast cancer risk associated with pathogenic BRCA1 variants?",
    "How does BRCA1 interact with BRCA2 in homologous recombination?",
]

research_log = []   # (question, answer) pairs — used for the consolidation step below
for i, q in enumerate(research_questions, 1):
    print(f"\n{'=' * 70}\nQ{i}: {q}\n{'=' * 70}")
    result = await Runner.run(research_agent, q, session=session)
    print(result.final_output)
    research_log.append((q, result.final_output))


Q1: What is BRCA1 and what is its primary function in DNA repair?


OPENAI_API_KEY is not set, skipping trace export


OPENAI_API_KEY is not set, skipping trace export


OPENAI_API_KEY is not set, skipping trace export


OPENAI_API_KEY is not set, skipping trace export


...

OPENAI_API_KEY is not set, skipping trace export


OPENAI_API_KEY is not set, skipping trace export


OPENAI_API_KEY is not set, skipping trace export


OPENAI_API_KEY is not set, skipping trace export


## How BRCA1 and BRCA2 Interact in Homologous Recombination

A key point that often surprises people: **BRCA1 and BRCA2 do not bind each other directly.** Instead, they are physically linked by a bridging scaffold protein, **PALB2** (Partner And Localizer of BRCA2), forming a functional **BRCA1–PALB2–BRCA2 axis** that executes homologous recombination (HR) repair of DNA double-strand breaks (DSBs) ([PMC – RAD51/PALB2 review](https://pmc.ncbi.nlm.nih.gov/articles/PMC12270392); [PMC – BRCA1-dependent recruitment](https://pmc.ncbi.nlm.nih.gov/articles/PMC9481714)).

### The Step-by-Step Relay

The two proteins act at **different, sequential stages** of the same pathway:

**1. BRCA1 — the upstream sensor and recruiter**
BRCA1 (as part of the BRCA1–BARD1 complex) is one of the first responders to a DSB. It:
- Helps **sense the break** and promotes **DNA end resection**, generating the 3′ single-stranded DNA (ssDNA) overhangs needed for HR. It does this in part by **countering 53BP1**, which

## 9. Inspect what the agent remembered — episodic vs semantic

The agent accumulated two distinct kinds of memory, and they live in different record types:

- **Episodic** — the conversation items replayed each turn, persisted by the `Session` protocol (stored tagged with the session's `thread_id`).
- **Semantic** — the durable findings the agent chose to save via `save_research_finding`, stored as `fact` records.

Let's look at both.

In [9]:
# EPISODIC: conversation items replayed each turn (via the Session protocol)
session_items = await session.get_items()
print(f"Episodic — session items: {len(session_items)}")
for it in session_items[:3]:
    print(f"  - {str(it)[:120]}...")

# SEMANTIC: durable findings the agent saved as `fact` records
findings = memory_client.search(
    "BRCA1 function and risk", user_id=USER_ID, agent_id=AGENT_ID,
    record_types=["fact"], max_results=10,
)
print(f"\nSemantic — durable research findings: {len(findings)}")
for r in findings:
    print(f"  - {r.content}")

Episodic — session items: 53
  - {'content': 'What is BRCA1 and what is its primary function in DNA repair?', 'role': 'user'}...
  - {'id': '__fake_id__', 'content': [{'annotations': [], 'text': "I'll research this question about BRCA1 and its role in D...
  - {'arguments': '{"query": "BRCA1 function DNA repair", "max_results": 5}', 'call_id': 'toolu_015pcJbqnqg9mvZHbZwSgyb1', '...

Semantic — durable research findings: 10
  - [BRCA1 loss consequences] Loss of BRCA1 function leads to defective homologous recombination, genomic instability, and increased cancer risk.
  - [BRCA1 breast cancer risk] Women with a pathogenic BRCA1 variant have an estimated cumulative breast cancer risk of approximately 72% (95% CI 65%–79%) by age 80, based on prospective cohort data (Kuchenbaecker et al., 2017).
  - [BRCA1 vs general population breast cancer risk] The ~72% lifetime breast cancer risk for BRCA1 carriers is roughly six times the ~12% lifetime risk in the general U.S. female population.
  - [B

oracleagentmemory/core/oracleagentmemory.py:853: UserWarning: You are calling an asynchronous method in a synchronous method from an asynchronous context. This is highly discouraged because it can lead to deadlocks. Please use the asynchronous method equivalent: 


## 9b. Reflect and synthesise — working memory

A deep-research agent's loop is *explore → gather → reflect → synthesise*. The web searches and saved findings above are the first two steps. The last two are **consolidation**: turning a sprawling investigation into a compact digest the next session can build on.

We replay the investigation onto a thread and let the memory layer do the reflection — `get_summary()` writes a research digest, and `get_context_card()` produces the prompt-ready working-memory block you'd seed the next session with.

> **💡 This completes the Deep Research memory profile.** **Episodic** (session items) + **semantic** (`fact` findings) + **working** (summary / context card) — the three tiers, each backed by the same Oracle store but accessed through a different interface.

In [10]:
from oracleagentmemory.apis.thread import Message

# WORKING memory: consolidate this investigation into a digest the next session can start from.
# We replay the Q&A onto a thread and let the memory layer summarise it (reflect -> synthesise).
digest_thread = memory_client.create_thread(
    thread_id="genome-digest-001", user_id=USER_ID, agent_id=AGENT_ID,
)
digest_msgs = []
for q, a in research_log:
    digest_msgs.append(Message(role="user", content=q))
    digest_msgs.append(Message(role="assistant", content=a))
digest_thread.add_messages(digest_msgs)

summary = digest_thread.get_summary()
print("=== Research digest (get_summary) ===")
print(summary.content)

card = digest_thread.get_context_card()
print("\n=== Context card — working-memory block to seed the next session ===")
print(card.formatted_content[:1000])

oracleagentmemory/core/thread.py:463: UserWarning: You are calling an asynchronous method in a synchronous method from an asynchronous context. This is highly discouraged because it can lead to deadlocks. Please use the asynchronous method equivalent: add_messages_async


oracleagentmemory/core/thread.py:792: UserWarning: You are calling an asynchronous method in a synchronous method from an asynchronous context. This is highly discouraged because it can lead to deadlocks. Please use the asynchronous method equivalent: get_summary_async


=== Research digest (get_summary) ===
The thread covers BRCA1 biology in depth across three topics: (1) BRCA1 as a tumor suppressor gene whose primary function is homologous recombination (HR) repair of DNA double-strand breaks, including its roles in end resection, repair pathway choice over NHEJ, and RAD51 recruitment. (2) Lifetime breast cancer risk for pathogenic BRCA1 variant carriers, estimated at ~72% by age 80 (Kuchenbaecker et al., 2017), with early onset, ~44% ovarian cancer risk, and ~40% contralateral breast cancer risk. (3) The BRCA1–PALB2–BRCA2 molecular axis in HR, where BRCA1 acts upstream to recruit PALB2, which bridges to BRCA2, which loads RAD51 onto resected ssDNA to execute strand invasion.


oracleagentmemory/core/thread.py:707: UserWarning: You are calling an asynchronous method in a synchronous method from an asynchronous context. This is highly discouraged because it can lead to deadlocks. Please use the asynchronous method equivalent: get_context_card_async



=== Context card — working-memory block to seed the next session ===
<context_card>
  <topics>
    <topic>kuchenbaecker 2017 brca risk estimates</topic>
    <topic>brca1-palb2-brca2 axis</topic>
    <topic>brca1 lifetime breast cancer risk</topic>
    <topic>brca1 tumor suppressor function</topic>
    <topic>rad51 loading mechanism</topic>
    <topic>homologous recombination dna repair</topic>
    <topic>brca1 ovarian cancer risk</topic>
    <topic>parp inhibitor hrd sensitivity</topic>
  </topics>
  <summary>
    The thread covers BRCA1 biology in depth across three topics: (1) BRCA1 as a tumor suppressor gene whose primary function is homologous recombination (HR) repair of DNA double-strand breaks, including its roles in end resection, repair pathway choice, and RAD51 recruitment. (2) Lifetime breast cancer risk for pathogenic BRCA1 variant carriers, estimated at ~72% by age 80 (Kuchenbaecker et al., 2017), with early onset and ~44% ovarian cancer risk. (3) The BRCA1–PALB2–BRCA2 mo

## 10. Verify continuity — resume a fresh session with the same memory store

The real test of a memory-aware agent is whether it can pick up where a prior session left off. Let's create a *new* session (simulating a separate process or later day) and ask a question that builds on prior findings.

If the agent recalls BRCA1 findings from the previous run without re-searching, we've demonstrated end-to-end memory continuity.

> **💡 Why it works — search scope.** `recall_research_findings` searches `fact` records by `user_id`/`agent_id` with the default `exact_thread_match=False`. Semantic memory isn't locked to the session that created it, so a brand-new `session_id` still sees every prior finding.

In [11]:
# Simulate a fresh session (new session_id, but same user/agent)
FRESH_SESSION_ID = "genome-session-002"
fresh_session = OracleAgentMemorySession(
    session_id=FRESH_SESSION_ID, client=memory_client,
    user_id=USER_ID, agent_id=AGENT_ID,
)

followup = "Based on what you already know about BRCA1, explain how a patient with a pathogenic BRCA1 variant might be counselled about screening."

print(f"FRESH SESSION — Q: {followup}\n")
result = await Runner.run(research_agent, followup, session=fresh_session)
print(result.final_output)

FRESH SESSION — Q: Based on what you already know about BRCA1, explain how a patient with a pathogenic BRCA1 variant might be counselled about screening.



## Counselling a Patient with a Pathogenic BRCA1 Variant

Here is how such a patient might be counselled, combining the risk picture with current screening and prevention guidance.

### 1. Frame the risk
A pathogenic BRCA1 variant means a substantially elevated lifetime cancer risk compared to the general population:

- **Breast cancer:** ~72% cumulative risk by age 80 (95% CI 65–79%) — roughly **six times** the ~12% general-population risk (Kuchenbaecker et al., 2017, *JAMA*).
- **Ovarian cancer:** ~44% lifetime risk by age 80.
- **Contralateral breast cancer:** ~40% within 20 years of a first breast cancer diagnosis.

It's important to emphasise these are population-level estimates; individual risk is modified by family history, age, and other factors.

### 2. Breast cancer screening (intensified surveillance)
Per [NCCN guidelines](https://densebreast-info.org/providers-faqs/what-is-the-screening-management-for-various-other-mutation-carriers) and [FORCE](https://www.facingourrisk.or

## 11. Cleanup (optional)

Uncomment the cell below to clear all session items and findings for this example. Leave it commented to keep the data for subsequent runs — that's often the point.

In [12]:
# await session.clear_session()
# await fresh_session.clear_session()
# for r in memory_client.search("BRCA", user_id=USER_ID, agent_id=AGENT_ID,
#                               record_types=["fact"], max_results=100):
#     memory_client.delete_memory(r.id)
# print("Cleaned up.")

connection.close()
print("Connection closed.")

Connection closed.


## Key Takeaways

> **🎯 1. The `Session` protocol is the clean integration point.** Four async methods (`get_items`, `add_items`, `pop_item`, `clear_session`) is all the OpenAI Agents SDK needs to plug a custom memory backend in. You don't have to modify the runner — you implement the protocol and pass an instance via `session=`.

> **🎯 2. Deep research uses three memory tiers, each a different record/interface.** **Episodic** = the `Session` (conversation replay). **Semantic** = durable findings saved as `fact` records and recalled with a `record_types` filter. **Working** = the thread's `get_summary()` / `get_context_card()`, which consolidate an investigation into a digest. Keep them distinct; don't dump findings into the conversation log.

> **🎯 3. Instructions steer memory usage.** A deep-research agent must be explicitly told to *recall before researching* and *save durable conclusions*. Otherwise the LLM treats memory as optional decoration and the store fills up with nothing useful.

> **🎯 4. Recall returns `formatted_content`, and scope controls reach.** Feed the agent each hit's `formatted_content` (prompt-ready, type-tagged) rather than hand-formatted strings. `exact_thread_match=False` (the default) is what lets a fresh session see prior semantic findings — continuity is a scope property, not a manual replay.

> **🎯 5. One governed substrate, testable continuity.** Episodic items, semantic findings, and the working-memory digest all live in one Oracle database — one pool, one backup, one compliance review. A new `session_id` with the same `user_id`/`agent_id` reasoning over prior findings is the simplest end-to-end test of a memory-aware agent.